# 11.3 只調圖片接頭，梯度怎麼穿過固定文字核心？


給一張紅方塊，目標短描述是 `red square`。如果圖片特徵能分相關屬性，文字核心也會用這些詞，可以先只調接頭，讓視覺線索接到文字行為。本節先確認答案代價能穿過固定核心回到接頭。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.data import ByteTokenizer
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss
from tiny_perceptron.multimodal import MultiModalLM, scene

torch.manual_seed(0)
tok = ByteTokenizer()
model = MultiModalLM(TinyLM(ModelConfig(width=8)))
model.requires_grad_(False)
model.image_projector.requires_grad_(True)
answer = tok.encode("red square") + [tok.eos_id]
prefix = [tok.bos_id, tok.user_id, tok.image_id, tok.eos_id, tok.assistant_id]
ids = torch.tensor(prefix + answer)
labels = torch.tensor([-100] * len(prefix) + answer)
out = model(ids, labels, image=scene())
loss = masked_loss(out["logits"], out["labels"])
loss.backward()
print("接頭收到梯度", model.image_projector.weight.grad.norm().item() > 0)
print("語言權重未累積梯度", model.language.embedding.weight.grad is None)

答案 10 個 ASCII 位元組加 EOS，共 11 個有效目標。前綴放角色、圖片與助手位置，labels 把前綴設 −100；模型先展開圖片，再做下一位置對齊。masked_loss 只對回答與結束算交叉熵，圖片仍是上下文。

兩行預期 True：接頭收到非零梯度，凍結的文字 embedding 沒有自己的 grad。固定文字運算仍傳遞對接頭的影響；沒有優化器 step，所以這個短例是梯度通路，不是生成成功率。

既有接頭小實驗真正更新後，完整描述仍為 0/6，雖然標準答案上下文裡的預測代價降低。紅方塊只生成 `red`，缺了空白與 square；綠圓還會重複 e。這保留一個有用差別：訓練時計算下一字常用標準前文，叫 teacher forcing；生成時接自己的上一字，前面一錯，後面可能跟著偏離。

因此要分開核對兩端能力、接頭更新、完整回答與 EOS。這次弱底座只有部分文字題做對，不能拿失敗推出所有接頭方案都無效；接頭也無法補回圖片入口已丟掉的資訊。

練習把 `model.image_projector.requires_grad_(True)` 改成 `False`，從建立模型開始重新執行整段程式。在 `loss.backward()` 前印可訓練總數與 `loss.requires_grad`，預期為 `0` 和 `False`；接著 backward 會因代價沒有梯度路徑而報錯。這是在前向計算前關閉全部梯度來源的反例，應在啟動訓練前攔下。若在 `out = model(...)` 之後才凍結，已建立的計算圖仍在，這次 backward 可以執行。

<details>
<summary>回顧與查證</summary>

可回顧：[11.1尺寸與理解的區別](https://birdhackor.github.io/tiny-perceptron-vlm/11.1.html)、[11.2凍結名單](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)、[10.7展開後才錯位](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)、[W.6梯度與更新](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/projector.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
